In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error

import pandas as pd
import numpy as np

# Load dataset (Replace with your downloaded CSV path)
df = pd.read_parquet("data/hdb_resale_price.parquet")
pd.set_option('display.max_columns', None)

# View your DataFrame
df.head()

def parse_lease(lease_str):
    if pd.isna(lease_str):
        return 99.0
    parts = str(lease_str).split()
    years = int(parts[0])
    months = int(parts[2]) if 'month' in parts else 0
    return years + (months / 12.0)

df['remaining_lease_years'] = df['remaining_lease'].apply(parse_lease)

#df["floor_level"] = df["storey_range"].str.split(" ").str[0].astype(float)

# B. Decode Storey Range to Numerical Midpoint (e.g., '10 TO 12' -> 11.0)
df['storey_midpoint'] = df['storey_range'].apply(lambda x: sum(map(int, str(x).split(' TO '))) / 2)

# C. Extract Numeric Time Values from Month (e.g., '2017-01' -> Year 2017, Month 1)
df['year'] = df['month'].apply(lambda x: int(str(x).split('-')[0]))
df['month_num'] = df['month'].apply(lambda x: int(str(x).split('-')[1]))

# features_3 = ["floor_area_sqm", "lease_commence_date", "floor_level"]
# X = df[features_3]
# y = df["resale_price"]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# baseline = LinearRegression()
# baseline.fit(X_train, y_train)
# preds = baseline.predict(X_test)

# mae_baseline = mean_absolute_error(y_test, preds)
# mape_baseline = mean_absolute_percentage_error(y_test, preds)
# r2_baseline = r2_score(y_test, preds)

# Part B — add flat_type and town
df["floor_level"] = df["storey_range"].str.split(" ").str[0].astype(float)
numeric = ["floor_area_sqm", "lease_commence_date", "storey_midpoint", "remaining_lease_years", "primary_schools_within_1km", "primary_schools_within_2km", 'malls_within_500m', 'malls_within_1km', 'mrt_within_500m',  'mrt_within_1km', "year", "month_num"]
category = ["flat_type", "town", "flat_model", "street_name"]


# 🔧 YOUR TURN: build X_more by combining the numeric + category columns,
#    then turning the categories into 0/1 columns.

X_more = pd.get_dummies(df[numeric + category], columns=category)   # <- fill the blank
y = df["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X_more, y, test_size=0.2, random_state=42)

# richer = LinearRegression()
# richer.fit(X_train, y_train)
# richer_preds = richer.predict(X_test)
# mae_richer = mean_absolute_error(y_test, richer_preds)
# mape_richer = mean_absolute_percentage_error(y_test, richer_preds)
# r2_richer = r2_score(y_test, richer_preds)

# forest = RandomForestRegressor(n_estimators=100, random_state=42)   # <- fill the blank
# forest.fit(X_train, y_train)
# mae_test   = mean_absolute_error(y_test,  forest.predict(X_test))
# mape_test  = mean_absolute_percentage_error(y_test,  forest.predict(X_test))
# r2_test    = r2_score(y_test,  forest.predict(X_test))


base_models = [
    ("random_forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("grad_boost",    GradientBoostingRegressor(random_state=42)),
]

# 🔧 YOUR TURN: the "manager" that learns how to blend the base models.
#    Use a simple LinearRegression() as the final_estimator.
stack = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),          # <- fill the blank
)
stack.fit(X_train, y_train)
stack_preds = stack.predict(X_test)
mae_stack  = mean_absolute_error(y_test, stack_preds)
mape_stack = mean_absolute_percentage_error(y_test, stack_preds)
r2_stack   = r2_score(y_test, stack_preds)

# Also score Gradient Boosting on its own, for a fair comparison
# gb = GradientBoostingRegressor(random_state=42)
# gb.fit(X_train, y_train)
# gb_preds = gb.predict(X_test)
# mae_gb  = mean_absolute_error(y_test, gb_preds)
# mape_gb = mean_absolute_percentage_error(y_test, gb_preds)
# r2_gb   = r2_score(y_test, gb_preds)

# print(f"Baseline (3 features): MAE S${mae_baseline:,.0f}   MAPE {mape_baseline:.1%}   R² {r2_baseline:.3f}")
# print(f"Linear Regression (5 features): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
# print(f"Random Forest (5 features): MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
# print()
# print(f"Gradient Boosting (5 features) : MAE S${mae_gb:,.0f}   MAPE {mape_gb:.1%}   R² {r2_gb:.3f}")
print(f"Stacked team  (Random Forest + Gradient Boosting) (5 features) : MAE S${mae_stack:,.0f}   MAPE {mape_stack:.1%}   R² {r2_stack:.3f}")